In [1]:
"""
Solutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.8 — Dimensionality Reduction)

Exercise 9: Load MNIST, train a RandomForest on raw data (time it), then on
            PCA-reduced data (95% explained variance) and compare speed +
            accuracy. Repeat with SGDClassifier.
Exercise 10: Use t-SNE to reduce the first 5,000 MNIST images to 2D and
             visualize. Compare against PCA, LLE, MDS.
"""


'\nSolutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.8 — Dimensionality Reduction)\n\nExercise 9: Load MNIST, train a RandomForest on raw data (time it), then on\n            PCA-reduced data (95% explained variance) and compare speed +\n            accuracy. Repeat with SGDClassifier.\nExercise 10: Use t-SNE to reduce the first 5,000 MNIST images to 2D and\n             visualize. Compare against PCA, LLE, MDS.\n'

In [2]:

import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, LocallyLinearEmbedding, MDS
from sklearn.metrics import accuracy_score

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# ---------------------------------------------------------------------------
# 0. Load MNIST, split 60k/10k as specified
# ---------------------------------------------------------------------------
def load_mnist_60k_10k():
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    X, y = mnist["data"], mnist["target"].astype(np.uint8)
    X_train, X_test = X[:60000], X[60000:]
    y_train, y_test = y[:60000], y[60000:]
    return X_train, y_train, X_test, y_test


In [4]:
# ---------------------------------------------------------------------------
# 9. PCA speedup for RandomForest and SGDClassifier
# ---------------------------------------------------------------------------
def exercise_9(X_train, y_train, X_test, y_test, random_state=42):
    # --- RandomForest on raw data ---
    rnd_clf = RandomForestClassifier(n_estimators=100, random_state=random_state)
    t0 = time.time()
    rnd_clf.fit(X_train, y_train)
    t1 = time.time()
    raw_train_time = t1 - t0
    print(f"Ex9 - RandomForest training time (raw): {raw_train_time:.2f}s")

    y_pred = rnd_clf.predict(X_test)
    raw_acc = accuracy_score(y_test, y_pred)
    print("Ex9 - RandomForest test accuracy (raw):", raw_acc)

    # --- PCA to 95% explained variance ---
    pca = PCA(n_components=0.95)
    t0 = time.time()
    X_train_reduced = pca.fit_transform(X_train)
    t1 = time.time()
    print(f"Ex9 - PCA fit_transform time: {t1 - t0:.2f}s, "
          f"reduced to {X_train_reduced.shape[1]} dimensions "
          f"(from {X_train.shape[1]})")

    # --- RandomForest on PCA-reduced data ---
    rnd_clf_pca = RandomForestClassifier(n_estimators=100, random_state=random_state)
    t0 = time.time()
    rnd_clf_pca.fit(X_train_reduced, y_train)
    t1 = time.time()
    pca_train_time = t1 - t0
    print(f"Ex9 - RandomForest training time (PCA): {pca_train_time:.2f}s")
    # Counterintuitive but expected result: RandomForest is often SLOWER on
    # PCA-reduced data. Tree-based models can exploit sparsity / find good
    # splits fast on the original high-dim pixel space (lots of near-zero
    # background pixels are cheap to split on); after PCA every feature is a
    # dense linear combination of pixels, which tends to *increase* the
    # effective work per split. PCA's speedup mainly helps distance-based or
    # linear models, not tree ensembles.

    X_test_reduced = pca.transform(X_test)
    y_pred_pca = rnd_clf_pca.predict(X_test_reduced)
    pca_acc = accuracy_score(y_test, y_pred_pca)
    print("Ex9 - RandomForest test accuracy (PCA):", pca_acc)
    # Also often slightly WORSE accuracy than on raw pixels, since PCA can
    # discard some fine-grained, low-variance-but-still-useful structure
    # that decision boundaries relied on.

    # --- SGDClassifier: raw vs PCA ---
    sgd_clf = SGDClassifier(random_state=random_state)
    t0 = time.time()
    sgd_clf.fit(X_train, y_train)
    t1 = time.time()
    sgd_raw_time = t1 - t0
    print(f"Ex9 - SGDClassifier training time (raw): {sgd_raw_time:.2f}s")
    sgd_raw_acc = accuracy_score(y_test, sgd_clf.predict(X_test))
    print("Ex9 - SGDClassifier test accuracy (raw):", sgd_raw_acc)

    sgd_clf_pca = SGDClassifier(random_state=random_state)
    t0 = time.time()
    sgd_clf_pca.fit(X_train_reduced, y_train)
    t1 = time.time()
    sgd_pca_time = t1 - t0
    print(f"Ex9 - SGDClassifier training time (PCA): {sgd_pca_time:.2f}s")
    sgd_pca_acc = accuracy_score(y_test, sgd_clf_pca.predict(X_test_reduced))
    print("Ex9 - SGDClassifier test accuracy (PCA):", sgd_pca_acc)
    # Unlike RandomForest, SGDClassifier (a linear model) typically DOES
    # train faster on the PCA-reduced data — fewer dimensions means fewer
    # weights to update each step — and accuracy is usually comparable or
    # even slightly better, since PCA's denoising can help a linear
    # decision boundary generalize.

    results = {
        "raw_forest": (raw_train_time, raw_acc),
        "pca_forest": (pca_train_time, pca_acc),
        "raw_sgd": (sgd_raw_time, sgd_raw_acc),
        "pca_sgd": (sgd_pca_time, sgd_pca_acc),
    }
    return pca, results


In [5]:
# ---------------------------------------------------------------------------
# 10. t-SNE visualization (first 5,000 MNIST images), compared to PCA/LLE/MDS
# ---------------------------------------------------------------------------
def plot_digits_scatter(X_2d, y, title, filepath, max_points=5000):
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        X_2d[:max_points, 0], X_2d[:max_points, 1],
        c=y[:max_points], cmap="tab10", s=8, alpha=0.6,
    )
    plt.legend(*scatter.legend_elements(), title="Digit", loc="best",
               bbox_to_anchor=(1.02, 1))
    plt.title(title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.tight_layout()
    plt.savefig(filepath, dpi=120)
    plt.close()
    print(f"Saved {title} plot to {filepath}")


def exercise_10(X_train, y_train, n_samples=5000, random_state=42,
                output_dir="."):
    X_subset = X_train[:n_samples]
    y_subset = y_train[:n_samples]

    # --- t-SNE ---
    t0 = time.time()
    tsne = TSNE(n_components=2, random_state=random_state, init="pca",
                learning_rate="auto")
    X_tsne = tsne.fit_transform(X_subset)
    t1 = time.time()
    print(f"Ex10 - t-SNE time: {t1 - t0:.2f}s")
    plot_digits_scatter(X_tsne, y_subset, "t-SNE on MNIST (5,000 samples)",
                         f"{output_dir}/tsne_mnist.png")

    # --- PCA (for comparison) ---
    t0 = time.time()
    pca = PCA(n_components=2, random_state=random_state)
    X_pca = pca.fit_transform(X_subset)
    t1 = time.time()
    print(f"Ex10 - PCA time: {t1 - t0:.2f}s")
    plot_digits_scatter(X_pca, y_subset, "PCA on MNIST (5,000 samples)",
                         f"{output_dir}/pca_mnist.png")
    # Expect PCA to show much more overlapping, blob-like clusters — PCA
    # only captures directions of maximum linear variance, not the
    # nonlinear manifold structure that actually separates digit classes.

    # --- LLE (for comparison) ---
    t0 = time.time()
    lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10,
                                  random_state=random_state)
    X_lle = lle.fit_transform(X_subset)
    t1 = time.time()
    print(f"Ex10 - LLE time: {t1 - t0:.2f}s")
    plot_digits_scatter(X_lle, y_subset, "LLE on MNIST (5,000 samples)",
                         f"{output_dir}/lle_mnist.png")

    # --- MDS (for comparison; slow — consider subsampling further, e.g. 2000) ---
    t0 = time.time()
    mds = MDS(n_components=2, random_state=random_state, n_init=1,
              normalized_stress="auto", init="random")
    X_mds = mds.fit_transform(X_subset[:2000])  # MDS scales poorly, subsample more
    t1 = time.time()
    print(f"Ex10 - MDS time: {t1 - t0:.2f}s")
    plot_digits_scatter(X_mds, y_subset[:2000], "MDS on MNIST (2,000 samples)",
                         f"{output_dir}/mds_mnist.png")

    # Expected takeaway: t-SNE typically gives the cleanest, most tightly
    # separated clusters per digit class (that's exactly what it's
    # optimized for — preserving local neighborhood structure), at the cost
    # of being by far the slowest and having no simple way to transform new
    # points (no .transform() method, unlike PCA/LLE).
    return X_tsne, X_pca, X_lle, X_mds


if __name__ == "__main__":
    print("Loading MNIST (requires network access to openml.org)...")
    X_train, y_train, X_test, y_test = load_mnist_60k_10k()

    print("\n=== Exercise 9 ===")
    pca, results = exercise_9(X_train, y_train, X_test, y_test)
    print("\nSummary:", results)

    print("\n=== Exercise 10 ===")
    exercise_10(X_train, y_train)


Loading MNIST (requires network access to openml.org)...

=== Exercise 9 ===
Ex9 - RandomForest training time (raw): 23.36s
Ex9 - RandomForest test accuracy (raw): 0.9691
Ex9 - PCA fit_transform time: 0.58s, reduced to 154 dimensions (from 784)
Ex9 - RandomForest training time (PCA): 57.13s
Ex9 - RandomForest test accuracy (PCA): 0.9487
Ex9 - SGDClassifier training time (raw): 115.64s
Ex9 - SGDClassifier test accuracy (raw): 0.8674
Ex9 - SGDClassifier training time (PCA): 18.90s
Ex9 - SGDClassifier test accuracy (PCA): 0.8931

Summary: {'raw_forest': (23.359423875808716, 0.9691), 'pca_forest': (57.12857389450073, 0.9487), 'raw_sgd': (115.64289593696594, 0.8674), 'pca_sgd': (18.903845071792603, 0.8931)}

=== Exercise 10 ===
Ex10 - t-SNE time: 13.05s
Saved t-SNE on MNIST (5,000 samples) plot to ./tsne_mnist.png
Ex10 - PCA time: 0.06s
Saved PCA on MNIST (5,000 samples) plot to ./pca_mnist.png
Ex10 - LLE time: 3.07s
Saved LLE on MNIST (5,000 samples) plot to ./lle_mnist.png
Ex10 - MDS time